# Лабораторна робота №6 — навчання з підкріпленням

**Тема:** Q-learning. **Мета:** дослідити оновлення Q-таблиці в грі «камінь–ножиці–папір» і перевірити алгоритм, яким керується робот у модифікованому `8_6.py`.

Робот у початковій програмі моделюється засобами NodeBox for OpenGL і Pymunk. Наведена нижче дискретна демонстрація перевіряє *лише алгоритм* навчання, а не фізику середовища.

## 1. Приклад Q-learning: «камінь–ножиці–папір»

Як у прикладі викладача: нічия дає винагороду 0.5, перемога — 1, поразка — 0. Застосовуємо правило часової різниці з $\alpha=0.1$ та $\gamma=0.9$. Для відтворюваності використовується 20 000 кроків зі сталим зерном генератора.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'rl_agent.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'rl_agent.py').is_file():
    raise FileNotFoundError('Очікується репозиторій із файлом rl_agent.py')
sys.path.insert(0, str(ROOT))
from rl_agent import QLearningAgent
OUT = ROOT / 'lab06' / 'results'
OUT.mkdir(parents=True, exist_ok=True)
print('Шлях до результатів:', OUT)

In [ ]:
actions = ['rock', 'paper', 'scissors']
Q = {s: {a: 0.0 for a in actions} for s in actions}
rng = random.Random(9)
alpha, gamma = 0.1, 0.9
beats = {'rock': 'scissors', 'paper': 'rock', 'scissors': 'paper'}
for _ in range(20000):
    state = rng.choice(actions)
    action = rng.choice(actions)
    reward = 0.5 if action == state else (1.0 if beats[action] == state else 0.0)
    next_state = action
    Q[state][action] += alpha * (reward + gamma * max(Q[next_state].values()) - Q[state][action])

for state in actions:
    best = max(Q[state], key=Q[state].get)
    print('Стан:', state, '| оптимальна дія:', best, '| Q:', {a: round(Q[state][a], 3) for a in actions})
values = np.array([[Q[s][a] for a in actions] for s in actions])
fig, ax = plt.subplots(figsize=(6,4))
im = ax.imshow(values, cmap='viridis')
ax.set(xticks=range(3), yticks=range(3), xticklabels=actions, yticklabels=actions,
       xlabel='Дія', ylabel='Стан суперника', title='Q-таблиця: камінь–ножиці–папір')
for i in range(3):
    for j in range(3):
        ax.text(j, i, '%.2f' % values[i,j], ha='center', va='center', color='white')
fig.colorbar(im, ax=ax, label='Q-value')
fig.tight_layout()
fig.savefig(OUT / '01_rock_paper_scissors_Q.png', dpi=170)
plt.show()

## 2. Удосконалена політика робота

Для робота є чотири стани: нічого не виявлено; корисний об’єкт; небажаний об’єкт; межа арени. Дві дії: зберегти напрямок або змінити напрямок.

Алгоритм реалізований в `rl_agent.py` і використовується в `8_6.py`. Він оновлює **попередню** пару «стан–дія» після надходження винагороди і нового стану. Рішення обираються за політикою $\varepsilon$-greedy зі спаданням $\varepsilon$.

In [ ]:
from lab06.demo_q_learning import train
agent, history = train(steps=10000, seed=9)
labels = ['нічого', 'корисний', 'небажаний', 'межа']
for s, label in enumerate(labels):
    print('%-12s Q=%s, найкраща дія: %d' % (label, [round(v,3) for v in agent.q[s]], agent.greedy_action(s)))
print('Кроків:', agent.steps, '| оновлень:', agent.updates)
assert [agent.greedy_action(i) for i in (1,2,3)] == [0,1,1]
print('Дискретний контрольний експеримент: PASS')

In [ ]:
q = np.array(agent.q)
fig, ax = plt.subplots(figsize=(6,4))
im = ax.imshow(q, cmap='viridis', aspect='auto')
ax.set(xticks=[0,1], xticklabels=['зберегти', 'змінити'],
       yticks=range(4), yticklabels=labels, xlabel='Дія', ylabel='Стан',
       title='Вивчена Q-таблиця робота (дискретний тест)')
for i in range(4):
    for j in range(2):
        ax.text(j,i,'%.2f'%q[i,j],color='white',ha='center',va='center')
fig.colorbar(im, ax=ax, label='Q-value')
fig.tight_layout()
fig.savefig(OUT/'02_robot_Q_table.png',dpi=170)
plt.show()
fig, ax = plt.subplots(figsize=(6,3.5))
ax.plot([h[0] for h in history], [h[1] for h in history], marker='o')
ax.set(xlabel='Кількість рішень', ylabel='epsilon', title='Зменшення інтенсивності дослідження')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT/'03_epsilon_decay.png',dpi=170)
plt.show()

## ВИСНОВОК

У прикладі «камінь–ножиці–папір» використано правило оновлення Q-таблиці Беллмана. Реалізовано й перевірено табличний Q-learning із коефіцієнтом навчання, дисконтуванням майбутньої винагороди та спадною ймовірністю дослідження. У **спрощеному контрольному експерименті** агент засвоїв бажані дії у станах виявлення об'єкта, небажаного об'єкта й наближення до межі. Реальну ефективність стратегії у фізичній симуляції потрібно оцінювати окремо у NodeBox/Pymunk.